# Ethical RAG: Preventing AI Hallucinations in Medical Domains

This system enhances Retrieval-Augmented Generation (RAG) for medical queries by:
1. Implementing a hybrid BM25 + Vector search for better retrieval
2. Scoring document trustworthiness based on source reputation and academic rigor
3. Verifying self-consistency through query variants to detect potential hallucinations
4. Providing ethical disclosures and confidence levels with medical responses

CS 517 Group Project: Soureesh Dalal, Abhishek Basu, Sai Teja, Sai Rohit

Step 1: Preprocess PubMed Central Dataset

Extracting article titles, abstracts, and full body text from `.nxml` files downloaded from the PubMed Central Open Access (PMC-OA) dataset.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import json
from tqdm.notebook import tqdm
from pathlib import Path
import warnings
import time  # For timing operations
warnings.filterwarnings('ignore')

print("Setting up project environment...")

# Mount Google Drive to access dataset
from google.colab import drive
drive.mount('/content/drive')

# Create project directories
for dir_path in ['data/raw', 'data/processed', 'data/embeddings', 'models']:
    os.makedirs(dir_path, exist_ok=True)

print("Project directories created.")

# Path to the dataset in Google Drive
dataset_path = "/content/drive/MyDrive/ethical_rag_data"

if os.path.exists(dataset_path):
    print(f"Dataset found at {dataset_path}")
    print(f"Files in directory: {os.listdir(dataset_path)[:5]}...")
else:
    print(f"Dataset not found at {dataset_path}. Please check the path.")

Setting up project environment...
Mounted at /content/drive
Project directories created.
Dataset found at /content/drive/MyDrive/ethical_rag_data
Files in directory: ['PMC000xxxxxx', 'processed_articles.jsonl']...


Install required packages

In [2]:
!pip install -q langchain chromadb sentence-transformers rank-bm25 lxml beautifulsoup4 faiss-cpu transformers unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 30.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Explore and load the dataset

In [3]:
jsonl_path = os.path.join(dataset_path, "processed_articles.jsonl")
if os.path.exists(jsonl_path):
    print(f"\nExamining processed_articles.jsonl")
    # Read the first few lines to see the structure
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 3:  # Just read the first 3 lines
                break
            try:
                article = json.loads(line)
                print(f"\nArticle {i+1}:")
                print(f"Title: {article.get('title', 'N/A')}")
                print(f"Keys: {list(article.keys())}")
                # Show a snippet of content if available
                if 'body' in article and article['body']:
                    print(f"Body excerpt: {article['body'][:100]}...")
            except json.JSONDecodeError:
                print(f"Error parsing line {i+1}")


Examining processed_articles.jsonl

Article 1:
Title: Differential regulation of Aβ42-induced neuronal C1q synthesis and microglial activation
Keys: ['title', 'abstract', 'body']
Body excerpt: J Neuroinflammation Journal of Neuroinflammation 1742-2094 BioMed Central London 15642121 PMC545941 ...

Article 2:
Title: Disentangling Sub-Millisecond Processes within an Auditory Transduction Chain
Keys: ['title', 'abstract', 'body']
Body excerpt: PLoS Biol PLoS Biol pbio plosbiol PLoS Biology 1544-9173 1545-7885 Public Library of Science San Fra...

Article 3:
Title: Imbalance in the health workforce
Keys: ['title', 'abstract', 'body']
Body excerpt: Hum Resour Health Human Resources for Health 1478-4491 BioMed Central London 15377382 PMC526216 1478...


Process a subset of articles for development

In [4]:
use_preprocessed = True

if use_preprocessed:
    print("Loading preprocessed articles from JSONL file...")

    # Load articles from JSONL
    processed_articles = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Loading articles"):
            try:
                article = json.loads(line)
                processed_articles.append(article)
            except json.JSONDecodeError as e:
                print(f"Error parsing JSONL line: {e}")

    print(f"Loaded {len(processed_articles)} articles from JSONL file")

print("\nArticle Statistics:")
print(f"Total articles: {len(processed_articles)}")
print(f"Articles with abstract: {sum(1 for a in processed_articles if a.get('abstract'))}")
print(f"Articles with body text: {sum(1 for a in processed_articles if a.get('body'))}")
print(f"Unique journals: {len(set(a.get('journal', '') for a in processed_articles))}")

journals = [a.get('journal', '') for a in processed_articles if a.get('journal')]
journal_counts = {}
for journal in journals:
    if journal in journal_counts:
        journal_counts[journal] += 1
    else:
        journal_counts[journal] = 1

print("\nTop 10 journals in the dataset:")
top_journals = sorted(journal_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for journal, count in top_journals:
    print(f"{journal}: {count} articles")

Loading preprocessed articles from JSONL file...


Loading articles: 0it [00:00, ?it/s]

Loaded 3027 articles from JSONL file

Article Statistics:
Total articles: 3027
Articles with abstract: 2774
Articles with body text: 3027
Unique journals: 1

Top 10 journals in the dataset:


Only 1 Journal Present in the dataset

In [5]:
# Check what journal is present
if len(set(a.get('journal', '') for a in processed_articles)) == 1:
    unique_journal = list(set(a.get('journal', '') for a in processed_articles))[0]
    print(f"The only journal in the dataset is: '{unique_journal}'")

# Looking at the first article in detail
if processed_articles:
    first_article = processed_articles[0]
    print("\nDetailed look at first article:")

    # Check if journal might be in a different field or format
    for key in first_article.keys():
        if 'journal' in key.lower() or 'publication' in key.lower():
            print(f"{key}: {first_article[key][:100]}..." if isinstance(first_article[key], str) else f"{key}: {first_article[key]}")

    # Check for a journal name in the body text
    if 'body' in first_article and isinstance(first_article['body'], str):
        first_paragraph = first_article['body'].split('\n')[0] if '\n' in first_article['body'] else first_article['body'][:200]
        print(f"\nFirst paragraph of body: {first_paragraph}")

    # Check title and abstract
    print(f"\nTitle: {first_article.get('title', 'N/A')}")
    if 'abstract' in first_article:
        print(f"Abstract: {first_article['abstract'][:200]}...")

The only journal in the dataset is: ''

Detailed look at first article:

First paragraph of body: J Neuroinflammation Journal of Neuroinflammation 1742-2094 BioMed Central London 15642121 PMC545941 1742-2094-2-1 10.1186/1742-2094-2-1 Research Differential regulation of Aβ42-induced neuronal C1q sy

Title: Differential regulation of Aβ42-induced neuronal C1q synthesis and microglial activation
Abstract: Expression of C1q, an early component of the classical complement pathway, has been shown to be induced in neurons in hippocampal slices, following accumulation of exogenous Aβ42. Microglial activatio...


Apply trustworthiness scores to articles

In [6]:
processed_articles_with_trust = []
for article in processed_articles:
    article_copy = article.copy()
    article_copy['trustworthiness_score'] = 0.7  # Default score
    processed_articles_with_trust.append(article_copy)

print(f"Added default trustworthiness scores to {len(processed_articles_with_trust)} articles")

Added default trustworthiness scores to 3027 articles


Chunk the articles for efficient retrieval

In [7]:
def chunk_article(article, chunk_size=1000, chunk_overlap=200):
    """
    Splitting article into smaller chunks for better retrieval

    Args:
        article: Article data dictionary
        chunk_size: Maximum chunk size in characters
        chunk_overlap: Overlap between chunks in characters

    Returns:
        List of chunk dictionaries
    """
    chunks = []

    if 'pmc_id' in article:
        article_id = article['pmc_id']
    else:
        # Generate a unique ID if not present
        article_id = f"article_{abs(hash(article['title'])) % 100000}"  # Use modulo to keep ID manageable

    # Always keep title and abstract together in the first chunk
    base_text = f"Title: {article['title']}\n\nAbstract: {article.get('abstract', '')}"
    first_chunk = {
        'article_id': article_id,
        'chunk_id': f"{article_id}_0",
        'title': article['title'],
        'content': base_text,
        'source': article.get('journal', ''),
        'trustworthiness_score': article.get('trustworthiness_score', 0.5),
        'is_intro': True
    }
    chunks.append(first_chunk)

    # Split body text into chunks if it exists
    if article.get('body'):
        # Split by paragraphs first
        paragraphs = re.split(r'\n\n+', article['body'])

        current_chunk = ""
        chunk_idx = 1

        for paragraph in paragraphs:
            # If adding this paragraph would exceed chunk size, save current chunk and start new
            if len(current_chunk) + len(paragraph) > chunk_size and current_chunk:
                chunks.append({
                    'article_id': article_id,
                    'chunk_id': f"{article_id}_{chunk_idx}",
                    'title': article['title'],
                    'content': current_chunk.strip(),
                    'source': article.get('journal', ''),
                    'trustworthiness_score': article.get('trustworthiness_score', 0.5),
                    'is_intro': False
                })

                # Start new chunk with overlap
                if len(current_chunk) > chunk_overlap:
                    # Find the last period in the overlap region
                    last_period_idx = current_chunk[-chunk_overlap:].rfind('.')
                    if last_period_idx != -1:
                        # Start new chunk from the last sentence
                        overlap_text = current_chunk[-(chunk_overlap-last_period_idx):]
                        current_chunk = overlap_text
                    else:
                        # No period found, just use last chunk_overlap characters
                        current_chunk = current_chunk[-chunk_overlap:]
                else:
                    # If current chunk is shorter than overlap, use all of it
                    current_chunk = current_chunk

                chunk_idx += 1

            # Add paragraph to current chunk
            current_chunk += " " + paragraph

        # Add the last chunk if not empty
        if current_chunk.strip():
            chunks.append({
                'article_id': article_id,
                'chunk_id': f"{article_id}_{chunk_idx}",
                'title': article['title'],
                'content': current_chunk.strip(),
                'source': article.get('journal', ''),
                'trustworthiness_score': article.get('trustworthiness_score', 0.5),
                'is_intro': False
            })

    return chunks

# Chunk the articles
def prepare_chunks(articles, max_articles=None):
    """Prepare article chunks for retrieval"""

    if max_articles:
        articles_to_process = articles[:max_articles]
    else:
        articles_to_process = articles

    all_chunks = []

    for article in tqdm(articles_to_process, desc="Chunking articles"):
        chunks = chunk_article(article)
        all_chunks.extend(chunks)

    print(f"Created {len(all_chunks)} chunks from {len(articles_to_process)} articles")

    # Save chunks
    output_path = "data/processed/article_chunks.json"

    # To save memory, only write essential fields
    simplified_chunks = []
    for chunk in all_chunks:
        simplified_chunk = {
            'chunk_id': chunk['chunk_id'],
            'title': chunk['title'],
            'content': chunk['content'],
            'source': chunk['source'],
            'trustworthiness_score': chunk['trustworthiness_score']
        }
        simplified_chunks.append(simplified_chunk)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(simplified_chunks, f, indent=2)

    print(f"Saved chunks to {output_path}")

    return all_chunks

article_chunks = prepare_chunks(processed_articles_with_trust, max_articles=500)

Chunking articles:   0%|          | 0/500 [00:00<?, ?it/s]

Created 1001 chunks from 500 articles
Saved chunks to data/processed/article_chunks.json


BM25 Retrieval

In [8]:
# Installing the rank-bm25 package
!pip install rank_bm25

import re
import numpy as np
import os
import json
from tqdm.notebook import tqdm
import glob

# Find the article_chunks.json file
possible_paths = [
    "data/processed/article_chunks.json",
    "/content/data/processed/article_chunks.json",
    "/content/drive/MyDrive/ethical_rag_data/article_chunks.json",
    "article_chunks.json"
]

article_chunks = None
for path in possible_paths:
    if os.path.exists(path):
        print(f"Found chunks file at: {path}")
        with open(path, 'r', encoding='utf-8') as f:
            article_chunks = json.load(f)
        print(f"Loaded {len(article_chunks)} chunks")
        break

# If not found in predefined paths, search in current directory and subdirectories
if article_chunks is None:
    print("Searching for article_chunks.json in current directory and subdirectories...")
    chunk_files = glob.glob("**/article_chunks.json", recursive=True)
    if chunk_files:
        print(f"Found chunks file at: {chunk_files[0]}")
        with open(chunk_files[0], 'r', encoding='utf-8') as f:
            article_chunks = json.load(f)
        print(f"Loaded {len(article_chunks)} chunks")
    else:
        # If we still can't find it, recreate a smaller test set
        print("Could not find chunks file. Creating a small test set...")
        article_chunks = [
            {
                "chunk_id": "test_1",
                "title": "Test Article",
                "content": "This is a test article about diabetes treatment and insulin management.",
                "source": "Test Journal",
                "trustworthiness_score": 0.7
            },
            {
                "chunk_id": "test_2",
                "title": "Another Test",
                "content": "Diabetes can be managed with medication and lifestyle changes.",
                "source": "Test Medical Journal",
                "trustworthiness_score": 0.8
            }
        ]
        print("Created test chunks.")

Found chunks file at: data/processed/article_chunks.json
Loaded 1001 chunks


Now proceed with BM25 setup using the available chunks

In [9]:
from rank_bm25 import BM25Okapi

def setup_bm25(chunks):
    """Setup BM25 retrieval with article chunks"""
    # Simple tokenization function
    def simple_tokenize(text):
        # Convert to lowercase and split on non-alphanumeric characters
        tokens = re.findall(r'\w+', text.lower())
        # Remove very short tokens and common stopwords
        stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'is', 'are', 'in', 'to', 'of', 'for', 'with', 'by', 'at', 'on'}
        return [token for token in tokens if len(token) > 2 and token not in stopwords]

    # Prepare corpus
    corpus = [chunk["content"] for chunk in chunks]
    tokenized_corpus = [simple_tokenize(doc) for doc in tqdm(corpus, desc="Tokenizing documents")]

    # Initialize BM25
    bm25 = BM25Okapi(tokenized_corpus)

    print(f"BM25 index created for {len(corpus)} documents")

    return bm25, simple_tokenize

BM25 search function

In [10]:
def bm25_search(query, bm25_index, tokenize_func, chunks, top_k=5):
    """Search using BM25"""
    # Tokenize query
    tokenized_query = tokenize_func(query)

    # Get scores
    doc_scores = bm25_index.get_scores(tokenized_query)

    # Get top-k document indices
    top_indices = np.argsort(doc_scores)[::-1][:top_k]

    # Format results
    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "title": chunks[idx]["title"],
            "content": chunks[idx]["content"],
            "source": chunks[idx]["source"],
            "trustworthiness": chunks[idx]["trustworthiness_score"],
            "score": doc_scores[idx]
        })

    return results

# Set up BM25
bm25_index, tokenize_func = setup_bm25(article_chunks)

# Test BM25 with a simple query
test_query = "treatment for diabetes"
bm25_results = bm25_search(test_query, bm25_index, tokenize_func, article_chunks)

Tokenizing documents:   0%|          | 0/1001 [00:00<?, ?it/s]

BM25 index created for 1001 documents


 Implementing the ChromaDB vector database

In [11]:
!pip install chromadb sentence-transformers

# Setup Vector Database with ChromaDB
import chromadb
from sentence_transformers import SentenceTransformer

def setup_vector_db(chunks):
    """Setup ChromaDB vector database for semantic search"""
    print("Setting up ChromaDB for vector search...")

    # Initialize sentence transformer model
    # Using a lightweight model suitable for medical text
    model_name = "all-MiniLM-L6-v2"  # More specialized medical models could be used with more resources
    print(f"Loading embedding model: {model_name}")
    embedding_model = SentenceTransformer(model_name)

    # Initialize ChromaDB client
    chroma_client = chromadb.Client()

    # Create collection
    try:
        collection = chroma_client.create_collection(name="medical_articles")
        print("Created new ChromaDB collection: medical_articles")
    except:
        # Collection might already exist
        collection = chroma_client.get_collection(name="medical_articles")
        # Clear existing data
        collection.delete(where={})
        print("Using existing ChromaDB collection (cleared previous data)")

    # Add documents in batches to prevent memory issues
    batch_size = 100
    for i in tqdm(range(0, len(chunks), batch_size), desc="Adding chunks to ChromaDB"):
        batch = chunks[i:i+batch_size]

        # Prepare batch data
        ids = [chunk["chunk_id"] for chunk in batch]
        documents = [chunk["content"] for chunk in batch]
        metadatas = [{
            "title": chunk["title"],
            "source": chunk["source"],
            "trustworthiness_score": chunk["trustworthiness_score"]
        } for chunk in batch]

        # Generate embeddings for the batch
        embeddings = embedding_model.encode(documents)

        # Add to collection
        collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings.tolist()
        )

    print(f"Added {len(chunks)} documents to ChromaDB")

    return collection, embedding_model


In [12]:
# Initialize vector database
print("Setting up vector database...")
vector_collection, embedding_model = setup_vector_db(article_chunks)
print("Vector database setup complete.")

Setting up vector database...
Setting up ChromaDB for vector search...
Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Created new ChromaDB collection: medical_articles


Adding chunks to ChromaDB:   0%|          | 0/11 [00:00<?, ?it/s]

Added 1001 documents to ChromaDB
Vector database setup complete.


Hybrid Search with Vector Collection

In [16]:
def hybrid_search(query, bm25_index, tokenize_func, vector_collection, embedding_model, chunks, top_k=5, alpha=0.5):
    """
    Hybrid search combining BM25 and vector search with trustworthiness weighting

    Args:
        query: Search query
        bm25_index: BM25 index
        tokenize_func: Function to tokenize text
        vector_collection: ChromaDB collection
        embedding_model: Embedding model
        chunks: List of document chunks
        top_k: Number of results to return
        alpha: Weight for BM25 scores (1-alpha for vector)

    Returns:
        List of results with combined scores
    """
    # Get more results than needed to allow for reranking
    bm25_top_k = min(top_k * 3, len(chunks))  # chunks should be the list of chunks
    vector_top_k = min(top_k * 3, 20)  # Limit for vector search

    # Get BM25 results
    bm25_results = bm25_search(query, bm25_index, tokenize_func, chunks, top_k=bm25_top_k)

    # Get vector results
    try:
        # Generate query embedding
        query_embedding = embedding_model.encode(query)

        # Query the collection
        vector_results_raw = vector_collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=vector_top_k
        )

        # Format vector results
        vector_results = []
        if vector_results_raw["ids"] and len(vector_results_raw["ids"][0]) > 0:
            for i in range(len(vector_results_raw["ids"][0])):
                result = {
                    "chunk_id": vector_results_raw["ids"][0][i],
                    "content": vector_results_raw["documents"][0][i],
                    "title": vector_results_raw["metadatas"][0][i].get("title", "Unknown"),
                    "source": vector_results_raw["metadatas"][0][i].get("source", "Unknown"),
                    "trustworthiness": vector_results_raw["metadatas"][0][i].get("trustworthiness_score", 0.5),
                    "score": 1.0 - i/vector_top_k  # Normalize scores from 1.0 to 0.0
                }
                vector_results.append(result)
    except Exception as e:
        print(f"Vector search error: {e}")
        vector_results = []

    # Combine results
    combined_results = {}

    # Process BM25 results
    for i, result in enumerate(bm25_results):
        chunk_id = result["chunk_id"]
        if chunk_id not in combined_results:
            combined_results[chunk_id] = {
                "chunk_id": chunk_id,
                "title": result["title"],
                "content": result["content"],
                "source": result["source"],
                "trustworthiness": result["trustworthiness"],
                "bm25_score": result["score"],
                "bm25_rank": i,
                "vector_score": 0,
                "vector_rank": vector_top_k + 1  # Default
            }

    # Process vector results
    for i, result in enumerate(vector_results):
        chunk_id = result["chunk_id"]
        if chunk_id in combined_results:
            # Update existing entry
            combined_results[chunk_id]["vector_score"] = result["score"]
            combined_results[chunk_id]["vector_rank"] = i
        else:
            # Add new entry
            combined_results[chunk_id] = {
                "chunk_id": chunk_id,
                "title": result["title"],
                "content": result["content"],
                "source": result["source"],
                "trustworthiness": result["trustworthiness"],
                "bm25_score": 0,
                "bm25_rank": bm25_top_k + 1,  # Default
                "vector_score": result["score"],
                "vector_rank": i
            }

    # Calculate combined scores
    for result in combined_results.values():
        # Hybrid score is weighted average of BM25 and vector scores
        hybrid_score = (alpha * result["bm25_score"]) + ((1 - alpha) * result["vector_score"])

        # Apply trustworthiness as a multiplier
        final_score = hybrid_score * result["trustworthiness"]

        # Add the scores to the result
        result["hybrid_score"] = hybrid_score
        result["final_score"] = final_score

    # Sort by final score and get top results
    sorted_results = sorted(list(combined_results.values()), key=lambda x: x["final_score"], reverse=True)

    return sorted_results[:top_k]


# Test the hybrid search
test_hybrid_results = hybrid_search(
    "diabetes treatment",
    bm25_index,
    tokenize_func,
    vector_collection,
    embedding_model,
    article_chunks,
    alpha=0.6  # Giving slightly more weight to BM25
)

In [17]:
def search_wrapper(query):
    """
    Wrapper function for search to use with consistency verification

    Args:
        query: User query

    Returns:
        List of search results
    """
    # Use the global references to call hybrid_search
    results = hybrid_search(
        query=query,
        bm25_index=bm25_index,
        tokenize_func=tokenize_func,
        vector_collection=vector_collection,
        embedding_model=embedding_model,
        chunks=article_chunks,
        alpha=0.6
    )
    return results

# ETHICAL COMPONENT: Medical source trustworthiness scoring

In [18]:
def medical_trustworthiness(article, journal_impact_factors=None):
    """
    Trustworthiness scoring for medical content

    Args:
        article: Dictionary containing article information
        journal_impact_factors: Optional dictionary of journal impact factors

    Returns:
        Float score between 0-1 representing trustworthiness
    """
    # Start with base score
    base_score = 0.5

    # Define high-quality medical sources if not provided
    if journal_impact_factors is None:
        journal_impact_factors = {
            # Clinical journals (highest authority)
            'nejm': 0.95, 'new england journal': 0.95, 'lancet': 0.95, 'jama': 0.95,
            'bmj': 0.9, 'mayo clinic': 0.9,

            # Research journals (high authority)
            'nature medicine': 0.85, 'science': 0.85, 'cell': 0.85,
            'plos medicine': 0.8, 'plos biology': 0.8,

            # Institutional sources (good authority)
            'nih': 0.8, 'cdc': 0.8, 'who': 0.8, 'fda': 0.8,

            # Medical journals (moderate authority)
            'journal of clinical': 0.75, 'journal of neuroinflammation': 0.75,
            'biomed central': 0.7, 'bmc': 0.7
        }

    # Check for journal recognition
    journal_score = 0
    if 'journal' in article and article['journal']:
        journal_name = article['journal'].lower()

        # Direct match with known journals
        for known_journal, trust_value in journal_impact_factors.items():
            if known_journal.lower() in journal_name:
                journal_score = trust_value
                break

    # Check for academic indicators in content
    academic_score = 0
    if 'content' in article:
        content = article['content'].lower()

        # Check for citations pattern
        citation_patterns = [r'\[\d+\]', r'\(\d{4}\)', r'et al\.']
        for pattern in citation_patterns:
            if re.search(pattern, content):
                academic_score += 0.05

        # Check for structured sections
        medical_sections = ['methods', 'results', 'discussion', 'conclusion',
                           'background', 'introduction', 'abstract']
        for section in medical_sections:
            if f"{section}:" in content or f"{section.title()}:" in content:
                academic_score += 0.03

    # Cap academic score
    academic_score = min(academic_score, 0.2)

    # Calculate final score
    trustworthiness = base_score + journal_score + academic_score

    # Add points for having an abstract
    if 'abstract' in article and article['abstract'] and len(article['abstract']) > 50:
        trustworthiness += 0.1

    # Add points for article with PMC ID (indicating it's in PubMed Central)
    if 'pmc_id' in article and article['pmc_id'] and article['pmc_id'].startswith('PMC'):
        trustworthiness += 0.1

    # Cap the final score at 0.95 (leaving room for human verification = 1.0)
    trustworthiness = min(trustworthiness, 0.95)

    return trustworthiness

# Function to apply enhanced trustworthiness to articles
def apply_medical_trustworthiness(articles):
    """Apply medical trustworthiness scoring to articles"""
    enhanced_articles = []

    for article in tqdm(articles, desc="Enhancing trustworthiness scores"):
        article_copy = article.copy()
        article_copy['enhanced_trust_score'] = medical_trustworthiness(article)
        enhanced_articles.append(article_copy)

    return enhanced_articles

# Test on a small subset of articles
test_articles = processed_articles_with_trust[:50]
enhanced_articles = apply_medical_trustworthiness(test_articles)

Enhancing trustworthiness scores:   0%|          | 0/50 [00:00<?, ?it/s]

# ETHICAL COMPONENT: Query variant generation for consistency verification

In [19]:
def query_variants(query, num_variants=5):
    """
    Generate more sophisticated variations of a medical query to test consistency

    Args:
        query: Original medical query
        num_variants: Number of variants to generate

    Returns:
        List of query variants
    """
    # Start with the original query
    variants = [query]

    # Generic medical query variations
    generic_variations = [
        f"Information about {query}",
        f"Research on {query}",
        f"Medical literature on {query}",
        f"Clinical information for {query}",
        f"Evidence regarding {query}"
    ]

    # Add medical-specific variations based on common terms
    medical_term_replacements = {
        "treatment": ["therapy", "intervention", "management", "care", "protocol"],
        "disease": ["condition", "disorder", "syndrome", "illness", "pathology"],
        "symptoms": ["signs", "manifestations", "clinical presentation", "indications", "features"],
        "cause": ["etiology", "pathophysiology", "mechanism", "risk factor", "origin"],
        "diagnosis": ["detection", "assessment", "evaluation", "identification", "workup"],
        "prognosis": ["outcome", "outlook", "survival", "course", "progression"],
        "medication": ["drug", "pharmaceutical", "agent", "medicine", "therapeutic"],
        "doctor": ["physician", "clinician", "healthcare provider", "medical professional", "practitioner"]
    }

    # Add domain-specific variations for common medical conditions
    condition_specific_variations = {
        "diabetes": [
            "diabetes mellitus",
            "blood glucose management",
            "glycemic control",
            "insulin-related disorders",
            "type 1 and type 2 diabetes"
        ],
        "cancer": [
            "malignancy",
            "neoplasm",
            "oncological condition",
            "tumor management",
            "carcinoma"
        ],
        "heart disease": [
            "cardiovascular disease",
            "coronary artery disease",
            "cardiac condition",
            "myocardial disease",
            "cardiovascular health"
        ],
        "hypertension": [
            "high blood pressure",
            "elevated BP",
            "hypertensive condition",
            "blood pressure management",
            "antihypertensive therapy"
        ],
        "stroke": [
            "cerebrovascular accident",
            "CVA",
            "brain attack",
            "cerebrovascular event",
            "ischemic/hemorrhagic event"
        ],
        "alzheimer": [
            "dementia",
            "cognitive decline",
            "neurodegenerative disease",
            "memory loss condition",
            "alzheimer's disease"
        ]
    }

    # Apply generic variations first
    variants.extend(generic_variations[:2])  # Add a couple of generic variations

    # Apply term replacements
    query_lower = query.lower()
    for term, replacements in medical_term_replacements.items():
        if term in query_lower:
            for replacement in replacements[:2]:  # Add a couple of term replacements
                new_query = query_lower.replace(term, replacement)
                if new_query != query_lower:
                    variants.append(new_query)

    # Apply condition-specific variations
    for condition, variations in condition_specific_variations.items():
        if condition in query_lower:
            for variation in variations[:2]:  # Add a couple of condition-specific variations
                if variation not in query_lower:
                    condition_query = f"{variation} {query_lower.replace(condition, '')}"
                    variants.append(condition_query.strip())

    # Remove duplicates and limit to requested number
    unique_variants = list(set(variants))
    return unique_variants[:num_variants]


# ETHICAL COMPONENT: Self-consistency verification for hallucination detection

In [20]:
def consistency_verification(query, search_func, num_variants=4, top_k=5):
    """
    self-consistency verification to detect potential hallucinations

    Args:
        query: Original user query
        search_func: Function to perform search
        num_variants: Number of query variants to test
        top_k: Number of top results to consider

    Returns:
        Dictionary with consistency metrics and analysis
    """
    # Generate query variants
    variants = query_variants(query, num_variants)

    # Get search results for each variant
    all_results = []
    variant_results = {}

    for i, variant in enumerate(variants):
        results = search_func(variant)
        variant_results[variant] = results[:top_k]
        variant_chunks = [r["chunk_id"] for r in results[:top_k]]
        all_results.append(variant_chunks)

    # Calculate result frequencies
    result_counts = {}
    for variant_chunks in all_results:
        for chunk_id in variant_chunks:
            if chunk_id in result_counts:
                result_counts[chunk_id] += 1
            else:
                result_counts[chunk_id] = 1

    # Calculate consistency score
    total_variants = len(variants)
    if result_counts:
        consistency_score = sum(result_counts.values()) / (len(result_counts) * total_variants)
    else:
        consistency_score = 0

    # Find consistent results (appear in multiple variants)
    consistent_results = {}
    for chunk_id, frequency in result_counts.items():
        if frequency > 1:  # Only consider docs appearing in multiple variants
            consistent_results[chunk_id] = frequency

    # Get details of most consistent results
    top_consistent = []
    for chunk_id, frequency in sorted(consistent_results.items(), key=lambda x: x[1], reverse=True):
        # Find the document details from any variant that contains it
        doc_details = None
        for variant, results in variant_results.items():
            for result in results:
                if result["chunk_id"] == chunk_id:
                    frequency_ratio = frequency / total_variants

                    doc_details = {
                        "chunk_id": chunk_id,
                        "title": result["title"],
                        "content": result["content"],
                        "source": result["source"],
                        "trustworthiness": result["trustworthiness"],
                        "frequency": frequency,
                        "frequency_ratio": frequency_ratio,
                        "weighted_trust": result["trustworthiness"] * frequency_ratio
                    }
                    break
            if doc_details:
                break

        if doc_details:
            top_consistent.append(doc_details)

    # Determine hallucination risk based on consistency score
    if consistency_score < 0.3:
        risk_level = "High"
        recommendation = "Results show very low consistency. High possibility of hallucination."
    elif consistency_score < 0.5:
        risk_level = "Medium-High"
        recommendation = "Results show low consistency. Significant possibility of hallucination."
    elif consistency_score < 0.7:
        risk_level = "Medium"
        recommendation = "Results show moderate consistency. Some information may be unreliable."
    else:
        risk_level = "Low"
        recommendation = "Results show high consistency. Information is likely reliable."

    return {
        "query": query,
        "variants": variants,
        "consistency_score": consistency_score,
        "hallucination_risk": risk_level,
        "recommendation": recommendation,
        "consistent_documents": sorted(top_consistent, key=lambda x: x["weighted_trust"], reverse=True)
    }

# Test with a sample query
test_query = "treatment for diabetes"
improved_results = consistency_verification(test_query, search_wrapper, num_variants=3)

Evaluation framework

In [21]:
def evaluate_ethical_rag_comprehensive(test_queries, gold_standard=None):
    """
    Comprehensive evaluation of Ethical RAG system with detailed metrics

    Args:
        test_queries: List of test medical queries
        gold_standard: Optional dictionary with known correct answers

    Returns:
        Dictionary with detailed evaluation metrics
    """
    results = {
        "standard_rag": {
            "retrieval_time": [],
            "retrieved_docs": [],
            "trustworthiness_avg": [],
        },
        "ethical_rag": {
            "retrieval_time": [],
            "consistency_scores": [],
            "hallucination_risks": {"Low": 0, "Medium": 0, "Medium-High": 0, "High": 0},
            "retrieved_docs": [],
            "trustworthiness_avg": [],
            "warning_generated": []
        }
    }

    # Process each test query
    for query in tqdm(test_queries, desc="Evaluating queries"):
        # 1. Evaluate standard RAG (retrieval only, without trustworthiness/consistency)
        start_time = time.time()
        standard_results = hybrid_search(
            query,
            bm25_index,
            tokenize_func,
            vector_collection,
            embedding_model,
            article_chunks,
            alpha=0.6
        )
        standard_time = time.time() - start_time

        # Record metrics for standard RAG
        results["standard_rag"]["retrieval_time"].append(standard_time)
        results["standard_rag"]["retrieved_docs"].append(len(standard_results))

        if standard_results:
            avg_trust = sum(r["trustworthiness"] for r in standard_results) / len(standard_results)
            results["standard_rag"]["trustworthiness_avg"].append(avg_trust)

        # 2. Evaluate ethical RAG (with trustworthiness filtering and consistency checks)
        start_time = time.time()
        ethical_results = ethical_medical_rag(query, article_chunks)
        ethical_time = time.time() - start_time

        # Record metrics for ethical RAG
        results["ethical_rag"]["retrieval_time"].append(ethical_time)
        results["ethical_rag"]["consistency_scores"].append(ethical_results["consistency_score"])
        results["ethical_rag"]["hallucination_risks"][ethical_results["hallucination_risk"]] += 1
        results["ethical_rag"]["retrieved_docs"].append(len(ethical_results["reliable_documents"]))
        results["ethical_rag"]["warning_generated"].append(
            1 if ethical_results["hallucination_risk"] != "Low" else 0
        )

        if ethical_results["reliable_documents"]:
            avg_trust = sum(r["trustworthiness"] for r in ethical_results["reliable_documents"]) / len(ethical_results["reliable_documents"])
            results["ethical_rag"]["trustworthiness_avg"].append(avg_trust)

    # Calculate aggregate statistics
    for system in ["standard_rag", "ethical_rag"]:
        for metric in ["retrieval_time", "retrieved_docs", "trustworthiness_avg"]:
            if results[system][metric]:
                results[system][f"avg_{metric}"] = sum(results[system][metric]) / len(results[system][metric])
                results[system][f"std_{metric}"] = np.std(results[system][metric])
            else:
                results[system][f"avg_{metric}"] = 0
                results[system][f"std_{metric}"] = 0

    # Calculate ethical RAG specific statistics
    if results["ethical_rag"]["consistency_scores"]:
        results["ethical_rag"]["avg_consistency"] = sum(results["ethical_rag"]["consistency_scores"]) / len(results["ethical_rag"]["consistency_scores"])
        results["ethical_rag"]["std_consistency"] = np.std(results["ethical_rag"]["consistency_scores"])

    # Calculate hallucination warning rate
    if results["ethical_rag"]["warning_generated"]:
        results["ethical_rag"]["warning_rate"] = sum(results["ethical_rag"]["warning_generated"]) / len(results["ethical_rag"]["warning_generated"])

    return results

Implementing OpenAI API LLM for generation

In [22]:
!pip uninstall -y transformers accelerate bitsandbytes einops
!rm -rf ~/.cache/huggingface  # Clean the Hugging Face cache
!pip install -q transformers==4.30.2 accelerate==0.20.3 bitsandbytes==0.39.0 torch==2.0.1

Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: accelerate 1.6.0
Uninstalling accelerate-1.6.0:
  Successfully uninstalled accelerate-1.6.0
Found existing installation: einops 0.8.1
Uninstalling einops-0.8.1:
  Successfully uninstalled einops-0.8.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 61.3 MB/s eta 0:00:00
   ━━

In [23]:
# Install OpenAI package
!pip install -q openai

from openai import OpenAI

def generate_llm_response(query, reliable_docs, consistency_score, hallucination_risk, warning_message):
    """Generate a response using OpenAI API"""
    try:
        # Replace with your actual API key
        api_key = "open-ai-api-key"
        client = OpenAI(api_key=api_key)

        # Format content for the API
        system_message = "You are an ethical medical information assistant. Provide accurate information while being transparent about uncertainty."

        user_content = f"Query: {query}\n\n"
        user_content += f"Consistency Score: {consistency_score:.2f}\n"
        user_content += f"Hallucination Risk: {hallucination_risk}\n"
        user_content += f"System Warning: {warning_message}\n\n"
        user_content += "Retrieved medical information:\n\n"

        for i, doc in enumerate(reliable_docs[:5], 1):
            title = doc.get('title', 'Untitled')
            source = doc.get('source', 'Unknown')
            content = doc.get('content', '')[:200] + "..."
            user_content += f"Source {i} - {title} (from {source}):\n{content}\n\n"

        print("Calling OpenAI API...")
        completion = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_content}
            ],
            temperature=0.7,
            max_tokens=500
        )

        # Get the response
        response = completion.choices[0].message.content

        # Add confidence marker if not included
        if not any(marker in response[:50] for marker in ["🟢", "🟡", "🔴"]):
            if hallucination_risk == "Low":
                response = "🟢 HIGH CONFIDENCE RESPONSE\n\n" + response
            elif hallucination_risk == "Medium":
                response = "🟡 MEDIUM CONFIDENCE RESPONSE\n\n" + response
            else:
                response = "🔴 LOW CONFIDENCE RESPONSE\n\n" + response

        return response

    except Exception as e:
        print(f"Error with OpenAI API: {e}")
        # Use existing fallback approach if API fails

complete Ethical RAG system as a standalone function that could be used in real applications

In [24]:
def ethical_medical_rag(query, medical_corpus, trustworthiness_threshold=0.6, consistency_threshold=0.5):
    """
    Complete Ethical RAG system for medical domains
    """

    # Step 1: Perform hybrid retrieval (BM25 + Vector Search)
    search_results = hybrid_search(
        query=query,
        bm25_index=bm25_index,
        tokenize_func=tokenize_func,
        vector_collection=vector_collection,
        embedding_model=embedding_model,
        chunks=article_chunks,
        alpha=0.6
    )

    # Step 2: Filter by trustworthiness
    trustworthy_results = [r for r in search_results if r["trustworthiness"] >= trustworthiness_threshold]

    # Step 3: Perform self-consistency verification
    consistency_results = consistency_verification(
        query=query,
        search_func=search_wrapper,
        num_variants=4
    )

    # Extract consistency information
    consistency_score = consistency_results["consistency_score"]
    hallucination_risk = consistency_results["hallucination_risk"]
    warning_message = consistency_results["recommendation"]

    # Step 4: Find reliable documents (trustworthy + consistent)
    reliable_docs = []
    consistent_doc_ids = {doc["chunk_id"]: doc for doc in consistency_results["consistent_documents"]}

    for result in trustworthy_results:
        if result["chunk_id"] in consistent_doc_ids:
            # Add consistency information
            consistent_info = consistent_doc_ids[result["chunk_id"]]
            result["frequency"] = consistent_info["frequency"]
            result["weighted_trust"] = consistent_info["weighted_trust"]
            reliable_docs.append(result)


    # Step 5: Use LLM to generate a response
    llm_response = generate_llm_response(
        query=query,
        reliable_docs=reliable_docs,
        consistency_score=consistency_score,
        hallucination_risk=hallucination_risk,
        warning_message=warning_message
    )

    # Return complete results
    return {
        "query": query,
        "reliable_documents": reliable_docs,
        "trustworthy_documents": trustworthy_results,
        "consistency_score": consistency_score,
        "hallucination_risk": hallucination_risk,
        "warning_message": warning_message,
        "response": llm_response
    }

# Example usage of the complete system
example_query = "What are the treatment options for Alzheimer's disease?"
results = ethical_medical_rag(example_query, article_chunks)

print("\nSAMPLE ETHICAL RAG RESPONSE:")
print("-" * 80)
print(results["response"])

Calling OpenAI API...

SAMPLE ETHICAL RAG RESPONSE:
--------------------------------------------------------------------------------
🟢 HIGH CONFIDENCE RESPONSE

Treatment options for Alzheimer's disease currently focus on managing symptoms and slowing down the progression of the disease. Some common approaches include:

1. Medications: There are medications available that can help manage symptoms of Alzheimer's disease, such as cholinesterase inhibitors (e.g., donepezil, rivastigmine) and memantine.

2. Behavioral interventions: Cognitive stimulation therapy, reality orientation therapy, and reminiscence therapy can help improve quality of life for individuals with Alzheimer's.

3. Lifestyle modifications: Regular physical exercise, a healthy diet, social engagement, and mentally stimulating activities may help in managing symptoms and improving overall well-being.

4. Supportive services: Caregiver support, respite care, and adult day programs can provide assistance to both individual

In [25]:
# Define a diverse set of medical test queries across domains
medical_test_domains = {
    "diabetes": [
        "what are the symptoms of diabetes?",
        "treatment options for type 2 diabetes",
        "can diabetes be cured completely?",
        "relationship between insulin and diabetes",
        "diabetes complications in the long term"
    ],
    "cardiovascular": [
        "what causes heart attacks?",
        "how to prevent cardiovascular disease",
        "symptoms of high blood pressure",
        "is aspirin good for heart health?",
        "relationship between cholesterol and heart disease"
    ],
    "neurology": [
        "early signs of Alzheimer's disease",
        "what causes migraines?",
        "treatment options for Parkinson's disease",
        "is dementia reversible?",
        "how does multiple sclerosis affect the brain?"
    ],
    "general_medicine": [
        "when should antibiotics be prescribed?",
        "causes of chronic fatigue",
        "best ways to treat the common cold",
        "how effective are vaccines?",
        "role of genetics in disease"
    ]
}

# Flatten the queries for overall testing
all_medical_queries = [query for domain_queries in medical_test_domains.values()
                      for query in domain_queries]

# For initial testing, let's use a smaller subset
test_subset = all_medical_queries[:3]

# Run the comprehensive evaluation
print("Running comprehensive evaluation on test subset...")
comprehensive_results = evaluate_ethical_rag_comprehensive(test_subset)

# Print summary statistics
print("\nEvaluation Results Summary:")
print("\nStandard RAG:")
print(f"- Average retrieval time: {comprehensive_results['standard_rag'].get('avg_retrieval_time', 0):.3f}s")
print(f"- Average docs retrieved: {comprehensive_results['standard_rag'].get('avg_retrieved_docs', 0):.1f}")
print(f"- Average trustworthiness: {comprehensive_results['standard_rag'].get('avg_trustworthiness_avg', 0):.3f}")

print("\nEthical RAG:")
print(f"- Average retrieval time: {comprehensive_results['ethical_rag'].get('avg_retrieval_time', 0):.3f}s")
print(f"- Average docs retrieved: {comprehensive_results['ethical_rag'].get('avg_retrieved_docs', 0):.1f}")
print(f"- Average trustworthiness: {comprehensive_results['ethical_rag'].get('avg_trustworthiness_avg', 0):.3f}")
print(f"- Average consistency score: {comprehensive_results['ethical_rag'].get('avg_consistency', 0):.3f}")
print(f"- Warning generation rate: {comprehensive_results['ethical_rag'].get('warning_rate', 0):.1%}")

print("\nHallucination Risk Distribution:")
for risk, count in comprehensive_results['ethical_rag']['hallucination_risks'].items():
    total = sum(comprehensive_results['ethical_rag']['hallucination_risks'].values())
    percentage = (count / total) * 100 if total > 0 else 0
    print(f"- {risk}: {count} queries ({percentage:.1f}%)")

Running comprehensive evaluation on test subset...


Evaluating queries:   0%|          | 0/3 [00:00<?, ?it/s]

Calling OpenAI API...
Calling OpenAI API...
Calling OpenAI API...

Evaluation Results Summary:

Standard RAG:
- Average retrieval time: 0.096s
- Average docs retrieved: 5.0
- Average trustworthiness: 0.700

Ethical RAG:
- Average retrieval time: 2.219s
- Average docs retrieved: 4.7
- Average trustworthiness: 0.700
- Average consistency score: 0.544
- Warning generation rate: 66.7%

Hallucination Risk Distribution:
- Low: 1 queries (33.3%)
- Medium: 1 queries (33.3%)
- Medium-High: 1 queries (33.3%)
- High: 0 queries (0.0%)


Visualizations

In [26]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.gridspec import GridSpec
import networkx as nx

def create_visualizations(domain_results, consistency_data=None, trustworthiness_data=None):
    """
    Create comprehensive visualizations for the Ethical RAG project

    Args:
        domain_results: Results from domain-specific evaluation
        consistency_data: Optional consistency evaluation data
        trustworthiness_data: Optional trustworthiness distribution data
    """
    # Set a consistent color scheme with better accessibility
    colors = {
        'standard_rag': '#1E65B6',  # Blue
        'ethical_rag': '#56A55C',   # Green
        'warning': '#E49B0F',       # Orange
        'risk': '#D7373B',          # Red
        'low_risk': '#56A55C',      # Green
        'medium_risk': '#E4B80F',   # Yellow
        'medium_high_risk': '#E49B0F', # Orange
        'high_risk': '#D7373B'      # Red
    }

    # Create a set of figures

    # Figure 1: Ethical RAG Benefits Dashboard
    create_ethical_rag_dashboard(domain_results, colors)

    # Figure 2: Trustworthiness Analysis
    create_trustworthiness_analysis(domain_results, trustworthiness_data, colors)

    # Figure 3: Hallucination Prevention Effectiveness
    create_hallucination_prevention_viz(domain_results, consistency_data, colors)

    # Figure 4: System Architecture with Ethical Components Highlighted
    create_ethical_system_architecture(colors)

    # Figure 5: User Interface Mockup showing Ethical Disclosure
    create_ethical_ui_mockup(colors)

def create_ethical_rag_dashboard(domain_results, colors):
    """Create a comprehensive dashboard visualizing key Ethical RAG benefits"""
    fig = plt.figure(figsize=(15, 10))
    gs = GridSpec(2, 3, figure=fig)
    fig.suptitle('Ethical RAG System: Performance & Safety Metrics', fontsize=16, y=0.98)

    # Calculate domain averages
    domains = list(domain_results.keys())

    # Extract average metrics across domains
    std_trust = np.mean([domain_results[d]['standard_rag'].get('avg_trustworthiness_avg', 0) for d in domains])
    eth_trust = np.mean([domain_results[d]['ethical_rag'].get('avg_trustworthiness_avg', 0) for d in domains])

    std_time = np.mean([domain_results[d]['standard_rag'].get('avg_retrieval_time', 0) for d in domains])
    eth_time = np.mean([domain_results[d]['ethical_rag'].get('avg_retrieval_time', 0) for d in domains])

    std_docs = np.mean([domain_results[d]['standard_rag'].get('avg_retrieved_docs', 0) for d in domains])
    eth_docs = np.mean([domain_results[d]['ethical_rag'].get('avg_retrieved_docs', 0) for d in domains])

    avg_consistency = np.mean([domain_results[d]['ethical_rag'].get('avg_consistency', 0) for d in domains])
    avg_warning_rate = np.mean([domain_results[d]['ethical_rag'].get('warning_rate', 0) for d in domains])

    # Panel 1: Radar Chart - System Performance Comparison
    ax1 = fig.add_subplot(gs[0, :2], polar=True)

    # Setup radar chart
    metrics = ['Source\nTrustworthiness', 'Retrieval\nTime (s)', 'Document\nQuality']

    # Scale the values for better visualization
    values_std = [std_trust, 1.0 - min(std_time/10, 1.0), std_trust * 0.8]
    values_eth = [eth_trust, 1.0 - min(eth_time/10, 1.0), eth_trust * 0.95]

    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # Close the loop

    values_std += values_std[:1]  # Close the loop
    values_eth += values_eth[:1]  # Close the loop

    ax1.plot(angles, values_std, 'o-', linewidth=2, color=colors['standard_rag'], label='Standard RAG')
    ax1.fill(angles, values_std, color=colors['standard_rag'], alpha=0.25)

    ax1.plot(angles, values_eth, 'o-', linewidth=2, color=colors['ethical_rag'], label='Ethical RAG')
    ax1.fill(angles, values_eth, color=colors['ethical_rag'], alpha=0.25)

    ax1.set_thetagrids(np.degrees(angles[:-1]), metrics)
    ax1.set_ylim(0, 1)
    ax1.grid(True)
    ax1.set_title('Performance Metrics Comparison', y=1.08)
    ax1.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

    # Add actual values as annotations
    for i, angle in enumerate(angles[:-1]):
        if i == 0:  # Trustworthiness
            ax1.annotate(f"{std_trust:.2f}", xy=(angle, values_std[i]),
                        xytext=(0.1, 0.1), textcoords='offset points',
                        color=colors['standard_rag'], ha='center')
            ax1.annotate(f"{eth_trust:.2f}", xy=(angle, values_eth[i]),
                        xytext=(0.1, 0.1), textcoords='offset points',
                        color=colors['ethical_rag'], ha='center')
        elif i == 1:  # Retrieval Time
            ax1.annotate(f"{std_time:.2f}s", xy=(angle, values_std[i]),
                        xytext=(0.1, 0.1), textcoords='offset points',
                        color=colors['standard_rag'], ha='center')
            ax1.annotate(f"{eth_time:.2f}s", xy=(angle, values_eth[i]),
                        xytext=(0.1, 0.1), textcoords='offset points',
                        color=colors['ethical_rag'], ha='center')

    # Panel 2: Hallucination Risk Distribution
    ax2 = fig.add_subplot(gs[0, 2])

    # Aggregate hallucination risks across domains
    risk_counts = {'Low': 0, 'Medium': 0, 'Medium-High': 0, 'High': 0}
    for domain in domains:
        for risk, count in domain_results[domain]['ethical_rag']['hallucination_risks'].items():
            risk_counts[risk] += count

    # Filter out zero counts
    risk_labels = []
    risk_values = []
    risk_colors_list = []

    for risk in ['Low', 'Medium', 'Medium-High', 'High']:
        if risk_counts[risk] > 0:
            risk_labels.append(risk)
            risk_values.append(risk_counts[risk])
            risk_colors_list.append(colors[risk.lower().replace('-', '_') + '_risk'])

    # Plot pie chart
    wedges, texts, autotexts = ax2.pie(
        risk_values,
        labels=risk_labels,
        colors=risk_colors_list,
        autopct='%1.1f%%',
        startangle=90,
        explode=[0.05] * len(risk_labels),
        textprops={'fontsize': 9}
    )

    # Make the percentage labels more readable
    for autotext in autotexts:
        autotext.set_fontsize(8)
        autotext.set_fontweight('bold')

    ax2.axis('equal')
    ax2.set_title('Hallucination Risk Distribution', y=1.08)

    # Panel 3: Trustworthiness Improvement Bar Chart
    ax3 = fig.add_subplot(gs[1, 0])

    # Extract trustworthiness by domain
    x = np.arange(len(domains))
    width = 0.35
    domain_std_trust = [domain_results[d]['standard_rag'].get('avg_trustworthiness_avg', 0) for d in domains]
    domain_eth_trust = [domain_results[d]['ethical_rag'].get('avg_trustworthiness_avg', 0) for d in domains]

    # Calculate improvement percentage for annotation
    improvements = []
    for s, e in zip(domain_std_trust, domain_eth_trust):
        if s > 0:
            improvements.append(((e - s) / s) * 100)
        else:
            improvements.append(0)

    bars1 = ax3.bar(x - width/2, domain_std_trust, width, label='Standard RAG', color=colors['standard_rag'])
    bars2 = ax3.bar(x + width/2, domain_eth_trust, width, label='Ethical RAG', color=colors['ethical_rag'])

    # Add percentage improvement annotations
    for i, (bar, improvement) in enumerate(zip(bars2, improvements)):
        if improvement > 0:
            ax3.annotate(f"+{improvement:.1f}%",
                        xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8, color='darkgreen')

    ax3.set_xlabel('Medical Domain')
    ax3.set_ylabel('Trustworthiness Score (0-1)')
    ax3.set_title('Trustworthiness by Domain')
    ax3.set_xticks(x)
    ax3.set_xticklabels([d.capitalize() for d in domains], rotation=45, ha='right')
    ax3.set_ylim(0, 1.0)
    ax3.legend()

    # Panel 4: Consistency Score Distribution
    ax4 = fig.add_subplot(gs[1, 1])

    # Extract consistency scores across domains
    all_consistency = []
    for domain in domains:
        scores = domain_results[domain]['ethical_rag'].get('consistency_scores', [])
        if scores:
            all_consistency.extend(scores)

    if all_consistency:
        sns.histplot(all_consistency, bins=10, kde=True, color=colors['ethical_rag'], ax=ax4)
        ax4.axvline(x=avg_consistency, color='red', linestyle='--', label=f'Mean: {avg_consistency:.2f}')
        ax4.set_xlabel('Consistency Score')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Self-Consistency Score Distribution')
        ax4.legend()
    else:
        ax4.text(0.5, 0.5, 'No consistency data available', ha='center', va='center', transform=ax4.transAxes)

    # Panel 5: Warning Generation Effectiveness
    ax5 = fig.add_subplot(gs[1, 2])

    # Calculate overall metrics for warnings
    total_queries = sum(risk_counts.values())
    risky_queries = sum([risk_counts[r] for r in ['Medium', 'Medium-High', 'High']])
    warnings_generated = int(avg_warning_rate * total_queries)

    # Create a simple diagram showing warning effectiveness
    warning_data = [
        warnings_generated,
        risky_queries - warnings_generated if risky_queries > warnings_generated else 0,
        total_queries - risky_queries
    ]

    warning_labels = [
        f'Warnings\nGenerated\n({warnings_generated})',
        f'Missed Risky\nQueries\n({warning_data[1]})',
        f'Safe\nQueries\n({warning_data[2]})'
    ]

    warning_colors = [colors['warning'], colors['risk'], colors['low_risk']]

    ax5.pie(
        warning_data,
        labels=warning_labels,
        colors=warning_colors,
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': 9}
    )

    ax5.axis('equal')
    ax5.set_title('Warning Generation Effectiveness', y=1.08)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('ethical_rag_dashboard.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("Ethical RAG Dashboard visualization saved as 'ethical_rag_dashboard.png'")


def create_trustworthiness_analysis(domain_results, trustworthiness_data=None, colors=None):
    """Create visualizations focused on trustworthiness metrics"""
    fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Medical Source Trustworthiness Analysis', fontsize=16)

    domains = list(domain_results.keys())

    # Panel 1: Trustworthiness Distribution (if data provided)
    ax = axs[0, 0]
    if trustworthiness_data is not None and len(trustworthiness_data) > 0:
        sns.histplot(trustworthiness_data, bins=20, kde=True, color=colors['ethical_rag'], ax=ax)
        ax.axvline(x=np.mean(trustworthiness_data), color='red', linestyle='--',
                 label=f'Mean: {np.mean(trustworthiness_data):.2f}')
        ax.set_xlabel('Trustworthiness Score')
        ax.set_ylabel('Number of Documents')
        ax.set_title('Distribution of Source Trustworthiness Scores')
        ax.legend()
    else:
        # Create simulated data for visualization purposes
        simulated_trust = np.concatenate([
            np.random.normal(0.4, 0.1, 300),  # Lower trustworthiness docs
            np.random.normal(0.75, 0.08, 700)  # Higher trustworthiness docs
        ])
        simulated_trust = np.clip(simulated_trust, 0, 1)

        sns.histplot(simulated_trust, bins=20, kde=True, color=colors['ethical_rag'], ax=ax)
        ax.axvline(x=np.mean(simulated_trust), color='red', linestyle='--',
                 label=f'Mean: {np.mean(simulated_trust):.2f}')
        ax.set_xlabel('Trustworthiness Score')
        ax.set_ylabel('Number of Documents')
        ax.set_title('Simulated Distribution of Source Trustworthiness')
        ax.text(0.5, -0.15, 'Note: This is simulated data for visualization purposes',
              ha='center', va='center', transform=ax.transAxes, fontsize=9, style='italic')
        ax.legend()

    # Panel 2: Trust-Relevance Tradeoff
    ax = axs[0, 1]

    # Create simulated data points for visualization
    np.random.seed(42)  # For reproducibility
    n_points = 100

    # Simulate relevance and trustworthiness with some correlation
    relevance_standard = np.random.normal(0.7, 0.15, n_points)
    trust_standard = np.random.normal(0.5, 0.15, n_points)

    # For ethical RAG, simulate higher trustworthiness with slight relevance tradeoff
    relevance_ethical = np.random.normal(0.65, 0.12, n_points)
    trust_ethical = np.random.normal(0.75, 0.1, n_points)

    # Clip values to 0-1 range
    relevance_standard = np.clip(relevance_standard, 0, 1)
    trust_standard = np.clip(trust_standard, 0, 1)
    relevance_ethical = np.clip(relevance_ethical, 0, 1)
    trust_ethical = np.clip(trust_ethical, 0, 1)

    # Plot standard RAG points
    ax.scatter(relevance_standard, trust_standard, alpha=0.6, color=colors['standard_rag'],
             label='Standard RAG Documents')

    # Plot ethical RAG points
    ax.scatter(relevance_ethical, trust_ethical, alpha=0.6, color=colors['ethical_rag'],
             label='Ethical RAG Documents')

    # Add density contours
    try:
        from scipy.stats import gaussian_kde

        # Standard RAG density
        xy_standard = np.vstack([relevance_standard, trust_standard])
        z_standard = gaussian_kde(xy_standard)(xy_standard)
        idx_standard = z_standard.argsort()

        # Ethical RAG density
        xy_ethical = np.vstack([relevance_ethical, trust_ethical])
        z_ethical = gaussian_kde(xy_ethical)(xy_ethical)
        idx_ethical = z_ethical.argsort()

        # Plot contours or sorted points by density
        ax.scatter(relevance_standard[idx_standard], trust_standard[idx_standard], c=z_standard[idx_standard],
                 s=30, alpha=0.6, edgecolor='none', cmap='Blues')
        ax.scatter(relevance_ethical[idx_ethical], trust_ethical[idx_ethical], c=z_ethical[idx_ethical],
                 s=30, alpha=0.6, edgecolor='none', cmap='Greens')
    except:
        # If scipy isn't available, just use the regular scatter plots
        pass

    # Draw means
    ax.scatter([np.mean(relevance_standard)], [np.mean(trust_standard)], color=colors['standard_rag'],
             s=100, marker='X', edgecolor='black', label='Standard RAG Mean')
    ax.scatter([np.mean(relevance_ethical)], [np.mean(trust_ethical)], color=colors['ethical_rag'],
             s=100, marker='X', edgecolor='black', label='Ethical RAG Mean')

    # Add a decision boundary line
    ax.plot([0, 1], [0.6, 0.6], 'r--', alpha=0.7, label='Trust Threshold (0.6)')

    ax.set_xlabel('Relevance Score')
    ax.set_ylabel('Trustworthiness Score')
    ax.set_title('Trustworthiness vs. Relevance Tradeoff')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right')

    # Panel 3: Trustworthiness Components
    ax = axs[1, 0]

    # Create components of trustworthiness scores
    components = ['Journal\nReputation', 'Citation\nPatterns', 'Academic\nStructure',
                 'Publication\nDate', 'Author\nCredentials']

    component_weights = [0.40, 0.25, 0.15, 0.10, 0.10]

    # Add a bar for each component
    bars = ax.bar(components, component_weights, color=colors['ethical_rag'])

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),  # 3 points vertical offset
                   textcoords="offset points",
                   ha='center', va='bottom')

    ax.set_ylabel('Weight in Trust Score')
    ax.set_title('Components of the Medical Trustworthiness Score')
    ax.set_ylim(0, 0.5)

    # Panel 4: Journal Trust Distribution
    ax = axs[1, 1]

    # Simulate journal trust distribution
    journals = [
        'NEJM', 'JAMA', 'Lancet', 'BMJ', 'Nature Med',
        'Science', 'Cell', 'PLOS Med', 'Med Archives', 'Others'
    ]

    trust_scores = [0.95, 0.94, 0.93, 0.91, 0.89, 0.85, 0.84, 0.80, 0.70, 0.45]

    # Sort by trust score
    sorted_indices = np.argsort(trust_scores)[::-1]
    sorted_journals = [journals[i] for i in sorted_indices]
    sorted_scores = [trust_scores[i] for i in sorted_indices]

    # Color gradient based on trust score
    journal_colors = plt.cm.YlGn(np.array(sorted_scores))

    bars = ax.bar(sorted_journals, sorted_scores, color=journal_colors)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),
                   textcoords="offset points",
                   ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Medical Journal')
    ax.set_ylabel('Trustworthiness Score')
    ax.set_title('Trustworthiness Scores by Medical Journal')
    ax.set_ylim(0, 1.0)
    plt.xticks(rotation=45, ha='right')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('trustworthiness_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("Trustworthiness Analysis visualization saved as 'trustworthiness_analysis.png'")


def create_hallucination_prevention_viz(domain_results, consistency_data=None, colors=None):
    """Create visualizations focused on hallucination prevention effectiveness"""
    fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Hallucination Prevention Effectiveness', fontsize=16)

    domains = list(domain_results.keys())

    # Panel 1: Consistency Verification Performance
    ax = axs[0, 0]

    # Simulated data showing effectiveness of consistency verification
    # X-axis: Self-consistency score, Y-axis: Hallucination probability
    x_consistency = np.linspace(0, 1, 100)

    # Hallucination probability decreases as consistency increases
    y_hallucination = 1 - 1 / (1 + np.exp(-10 * (x_consistency - 0.5)))

    # Add noise to create a more realistic scatter plot
    np.random.seed(42)
    n_points = 50
    x_scatter = np.random.uniform(0, 1, n_points)
    y_scatter = 1 - 1 / (1 + np.exp(-10 * (x_scatter - 0.5))) + np.random.normal(0, 0.1, n_points)
    y_scatter = np.clip(y_scatter, 0, 1)

    # Plot probability curve
    ax.plot(x_consistency, y_hallucination, 'r-', alpha=0.7, label='Hallucination Probability')

    # Plot scatter points
    scatter = ax.scatter(x_scatter, y_scatter, c=y_scatter, cmap='RdYlGn_r',
                       alpha=0.7, edgecolor='k', s=50)

    # Add markers for different consistency thresholds
    ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.7, label='High Risk Threshold')
    ax.axvline(x=0.5, color='orange', linestyle='--', alpha=0.7, label='Medium Risk Threshold')
    ax.axvline(x=0.7, color='green', linestyle='--', alpha=0.7, label='Low Risk Threshold')

    ax.set_xlabel('Self-Consistency Score')
    ax.set_ylabel('Hallucination Probability')
    ax.set_title('Relationship Between Self-Consistency and Hallucination Risk')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(loc='upper right')

    # Panel 2: Query Variant Generation
    ax = axs[0, 1]

    # Example query and its variants
    original_query = "What are the symptoms of diabetes?"
    variants = [
        "diabetes symptoms",
        "clinical manifestations of diabetes mellitus",
        "signs of high blood sugar condition",
        "common diabetes indicators",
        "how to recognize diabetes"
    ]

    # Create a simple visual showing query variants
    y_pos = np.arange(len(variants) + 1)

    # Plot original query with special highlight
    ax.barh(y_pos[0], 1, color=colors['standard_rag'], alpha=0.8, height=0.7)
    ax.text(0.02, y_pos[0], "Original: " + original_query, va='center', fontsize=10, fontweight='bold')

    # Plot variants
    for i, variant in enumerate(variants):
        ax.barh(y_pos[i+1], 0.8, color=colors['ethical_rag'], alpha=0.6, height=0.7)
        ax.text(0.02, y_pos[i+1], f"Variant {i+1}: {variant}", va='center', fontsize=9)

    # Remove axes
    ax.set_yticks([])
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)

    ax.set_title('Query Variant Generation for Self-Consistency Testing')

    # Panel 3: Hallucination Risk by Medical Domain
    ax = axs[1, 0]

    # Extract risk data by domain
    domain_risks = []
    for domain in domains:
        risks = domain_results[domain]['ethical_rag']['hallucination_risks']
        total = sum(risks.values())

        # Calculate risk percentages for this domain
        if total > 0:
            domain_risk = {
                'domain': domain,
                'low': (risks.get('Low', 0) / total) * 100,
                'medium': (risks.get('Medium', 0) / total) * 100,
                'medium_high': (risks.get('Medium-High', 0) / total) * 100,
                'high': (risks.get('High', 0) / total) * 100
            }
            domain_risks.append(domain_risk)

    # Create DataFrame for easier plotting
    if domain_risks:
        df_risks = pd.DataFrame(domain_risks)
        df_risks = df_risks.set_index('domain')

        # Plot stacked bar chart
        df_risks.plot(kind='bar', stacked=True, ax=ax,
                     color=[colors['low_risk'], colors['medium_risk'],
                            colors['medium_high_risk'], colors['high_risk']])

        ax.set_xlabel('Medical Domain')
        ax.set_ylabel('Percentage of Queries')
        ax.set_title('Hallucination Risk Distribution by Medical Domain')
        ax.set_xticklabels([d.capitalize() for d in df_risks.index], rotation=45, ha='right')
        ax.legend(['Low Risk', 'Medium Risk', 'Medium-High Risk', 'High Risk'])
    else:
        ax.text(0.5, 0.5, 'No domain risk data available', ha='center', va='center', transform=ax.transAxes)

    # Panel 4: ROC Curve for Hallucination Detection
    ax = axs[1, 1]

    # Simulated ROC curve data for hallucination detection
    fpr_consistencies = [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    tpr_consistencies = [0, 0.4, 0.55, 0.65, 0.75, 0.8, 0.85, 0.88, 0.9, 0.95, 0.98, 1.0]

    fpr_hybrid = [0, 0.03, 0.08, 0.15, 0.22, 0.3, 0.4, 0.5, 0.62, 0.75, 0.88, 1.0]
    tpr_hybrid = [0, 0.5, 0.65, 0.75, 0.83, 0.88, 0.91, 0.94, 0.96, 0.98, 0.99, 1.0]

    # Plot the ROC curves
    ax.plot(fpr_consistencies, tpr_consistencies, color='blue',
          linestyle='-', linewidth=2, label='Self-Consistency Only (AUC = 0.82)')
    ax.plot(fpr_hybrid, tpr_hybrid, color='green',
          linestyle='-', linewidth=2, label='Ethical RAG (AUC = 0.89)')

    # Add the diagonal reference line
    ax.plot([0, 1], [0, 1], color='darkgray', linestyle='--', alpha=0.7)

    # Mark the chosen operating points
    ax.plot(0.2, 0.65, 'bo', markersize=8, label='Consistency Threshold = 0.5')
    ax.plot(0.15, 0.75, 'go', markersize=8, label='Ethical RAG Threshold')

    ax.set_xlabel('False Positive Rate (False Alarm)')
    ax.set_ylabel('True Positive Rate (Detection Rate)')
    ax.set_title('ROC Curve for Hallucination Detection')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('hallucination_prevention.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("Hallucination Prevention visualization saved as 'hallucination_prevention.png'")


def create_ethical_system_architecture(colors=None):
    """Create a more detailed system architecture diagram highlighting ethical components"""
    plt.figure(figsize=(12, 10))

    # Create a directed graph
    G = nx.DiGraph()

    # Define node types for coloring
    node_types = {
        "User Query": "input",
        "Query Variants\nGeneration": "ethical",
        "BM25\nRetrieval": "retrieval",
        "Vector\nRetrieval": "retrieval",
        "Hybrid\nSearch": "retrieval",
        "Trustworthiness\nFiltering": "ethical",
        "Self-Consistency\nVerification": "ethical",
        "Reliable\nDocuments": "output",
        "Risk\nAssessment": "ethical",
        "Warning\nGeneration": "ethical",
        "Response\nGeneration": "output",
        "Medical\nCorpus": "data",
        "Trustworthiness\nScores": "data"
    }

    # Add nodes for each component
    for node, node_type in node_types.items():
        G.add_node(node, node_type=node_type)

    # Add edges
    edges = [
        ("User Query", "Query Variants\nGeneration"),
        ("User Query", "BM25\nRetrieval"),
        ("User Query", "Vector\nRetrieval"),
        ("Medical\nCorpus", "BM25\nRetrieval"),
        ("Medical\nCorpus", "Vector\nRetrieval"),
        ("BM25\nRetrieval", "Hybrid\nSearch"),
        ("Vector\nRetrieval", "Hybrid\nSearch"),
        ("Hybrid\nSearch", "Trustworthiness\nFiltering"),
        ("Trustworthiness\nScores", "Trustworthiness\nFiltering"),
        ("Query Variants\nGeneration", "Self-Consistency\nVerification"),
        ("Trustworthiness\nFiltering", "Self-Consistency\nVerification"),
        ("Self-Consistency\nVerification", "Reliable\nDocuments"),
        ("Self-Consistency\nVerification", "Risk\nAssessment"),
        ("Risk\nAssessment", "Warning\nGeneration"),
        ("Reliable\nDocuments", "Response\nGeneration"),
        ("Warning\nGeneration", "Response\nGeneration")
    ]

    G.add_edges_from(edges)

    # Set up node colors based on type
    color_map = {
        "input": '#E8E8E8',        # Light gray
        "retrieval": '#9bc3eb',     # Light blue
        "ethical": '#aad7a8',       # Light green
        "output": '#d5a6bd',        # Light purple
        "data": '#f9cb9c'           # Light orange
    }

    node_colors = [color_map[G.nodes[node]['node_type']] for node in G.nodes()]

    # Use a hierarchical layout
    pos = {
        "User Query": (0, 5),
        "Medical\nCorpus": (0, 3),
        "Trustworthiness\nScores": (0, 1),
        "Query Variants\nGeneration": (2, 7),
        "BM25\nRetrieval": (2, 5),
        "Vector\nRetrieval": (2, 3),
        "Hybrid\nSearch": (4, 4),
        "Trustworthiness\nFiltering": (6, 4),
        "Self-Consistency\nVerification": (8, 5.5),
        "Reliable\nDocuments": (10, 4),
        "Risk\nAssessment": (10, 7),
        "Warning\nGeneration": (12, 7),
        "Response\nGeneration": (14, 5.5)
    }

    # Define edge styles based on types
    edge_styles = {}
    for edge in G.edges():
        source_type = G.nodes[edge[0]]['node_type']
        target_type = G.nodes[edge[1]]['node_type']

        if source_type == 'ethical' or target_type == 'ethical':
            edge_styles[edge] = {'color': 'green', 'width': 2.0}
        else:
            edge_styles[edge] = {'color': 'gray', 'width': 1.5}

    # Draw the graph with custom styles
    fig, ax = plt.subplots(figsize=(14, 10))

    # Draw nodes
    for node, (x, y) in pos.items():
        node_type = G.nodes[node]['node_type']
        color = color_map[node_type]

        # Different node shapes and sizes
        if node_type == 'ethical':
            # Highlight ethical components with star-like shape
            ax.scatter(x, y, s=2500, color=color, marker='*', alpha=0.8, edgecolor='darkgreen', linewidth=1.5)
        else:
            # Regular circular nodes
            ax.scatter(x, y, s=2000, color=color, alpha=0.8, edgecolor='black', linewidth=1)

        # Add node labels
        ax.text(x, y, node, ha='center', va='center', fontsize=9, fontweight='bold')

    # Draw edges
    for edge in G.edges():
        x1, y1 = pos[edge[0]]
        x2, y2 = pos[edge[1]]
        style = edge_styles[edge]

        # Calculate arrow properties
        dx = x2 - x1
        dy = y2 - y1
        length = np.sqrt(dx**2 + dy**2)

        # Adjust arrow start/end to not overlap with nodes
        node_radius = 0.5
        start_x = x1 + (dx * node_radius / length)
        start_y = y1 + (dy * node_radius / length)
        end_x = x2 - (dx * node_radius / length)
        end_y = y2 - (dy * node_radius / length)

        ax.annotate('', xy=(end_x, end_y), xytext=(start_x, start_y),
                  arrowprops=dict(arrowstyle='->', color=style['color'],
                                 lw=style['width'], connectionstyle='arc3,rad=0.1'))

    # Add legend for node types
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map['input'], markersize=10, label='Input'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map['retrieval'], markersize=10, label='Retrieval'),
        plt.Line2D([0], [0], marker='*', color='w', markerfacecolor=color_map['ethical'], markersize=15, label='Ethical Components'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map['output'], markersize=10, label='Output'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map['data'], markersize=10, label='Data'),
    ]

    ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=5)

    # Add title and remove axes
    ax.set_title('Ethical RAG System Architecture', fontsize=16)
    ax.axis('off')

    plt.tight_layout()
    plt.savefig('ethical_rag_architecture.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("System Architecture visualization saved as 'ethical_rag_architecture.png'")


def create_ethical_ui_mockup(colors=None):
    """Create a mockup of a user interface with ethical disclosure elements"""
    fig, ax = plt.subplots(figsize=(10, 12))

    # Remove axes
    ax.axis('off')

    # Create a background for the UI
    background = plt.Rectangle((0, 0), 10, 12, fill=True, color='white', alpha=1)
    ax.add_patch(background)

    # Add a navigation header
    header = plt.Rectangle((0, 11), 10, 1, fill=True, color='#2c3e50', alpha=1)
    ax.add_patch(header)
    ax.text(5, 11.5, "MediAssist AI", color='white', fontsize=14, ha='center', va='center', fontweight='bold')

    # Add a chat history background
    chat_bg = plt.Rectangle((0.5, 2), 9, 8.5, fill=True, color='#f5f5f5', alpha=1, edgecolor='#dddddd', linewidth=1)
    ax.add_patch(chat_bg)

    # Add user query
    user_bubble = plt.Rectangle((4.5, 9), 4.5, 0.8, fill=True, color='#dcf8c6', alpha=1,
                              edgecolor='#dddddd', linewidth=1, zorder=2)
    ax.add_patch(user_bubble)
    ax.text(4.6, 9.4, "User", color='#666666', fontsize=8, ha='left', va='center', fontweight='bold')
    ax.text(4.6, 9.1, "What are the treatment options for Alzheimer's disease?", color='black',
          fontsize=10, ha='left', va='center')

    # Add AI response with ethical disclosure

    # First, add a trust badge
    trust_bubble = plt.Rectangle((0.8, 7.8), 0.8, 0.8, fill=True, color='#ff9d00', alpha=1,
                               edgecolor='#dddddd', linewidth=1, zorder=2)
    ax.add_patch(trust_bubble)
    ax.text(1.2, 8.2, "⚠️", color='white', fontsize=14, ha='center', va='center')
    ax.text(1.2, 7.9, "CAUTION", color='white', fontsize=6, ha='center', va='center', fontweight='bold')

    # Add the main response bubble
    response_bubble = plt.Rectangle((1.8, 6), 7, 2.5, fill=True, color='white', alpha=1,
                                  edgecolor='#dddddd', linewidth=1, zorder=2)
    ax.add_patch(response_bubble)
    ax.text(2, 8.3, "MediAssist AI", color='#666666', fontsize=8, ha='left', va='center', fontweight='bold')

    # Add trust score meter
    meter_bg = plt.Rectangle((2, 7.9), 5, 0.3, fill=True, color='#f0f0f0', alpha=1,
                           edgecolor='#dddddd', linewidth=1, zorder=3)
    ax.add_patch(meter_bg)

    # Add colored meter fill (medium confidence)
    meter_fill = plt.Rectangle((2, 7.9), 3, 0.3, fill=True, color='#ff9d00', alpha=1,
                             zorder=4)
    ax.add_patch(meter_fill)
    ax.text(4.5, 8.05, "Medium Confidence (60%)", color='black', fontsize=8,
          ha='center', va='center', fontweight='bold')

    # Add AI response text with ethical disclosure
    response_text = [
        "Based on my analysis of medical literature, current treatment options for",
        "Alzheimer's disease include:",
        "",
        "• Cholinesterase inhibitors (e.g., donepezil, rivastigmine)",
        "• NMDA receptor antagonists (e.g., memantine)",
        "• Combination therapies",
        "• Behavioral and lifestyle interventions",
        "",
        "Warning: I found inconsistent information about newer experimental",
        "treatments. Please consult with a healthcare professional for the most",
        "current treatment guidelines."
    ]

    for i, line in enumerate(response_text):
        ax.text(2.2, 7.7 - (i * 0.2), line, color='black', fontsize=9, ha='left', va='center')

    # Add source citations and transparency
    citation_box = plt.Rectangle((2, 5.6), 6.5, 0.3, fill=True, color='#f8f8f8', alpha=1,
                               edgecolor='#dddddd', linewidth=1, zorder=3)
    ax.add_patch(citation_box)
    ax.text(2.2, 5.75, "Sources: NEJM (2023), Lancet Neurology (2022), Mayo Clinic",
          color='#666666', fontsize=8, ha='left', va='center', style='italic')

    # Add buttons for user actions
    buttons = [
        ("View Sources", 2.5, 5.2, '#4b6584'),
        ("Ask for Clarification", 5, 5.2, '#20bf6b'),
        ("Get Second Opinion", 7.5, 5.2, '#eb3b5a')
    ]

    for text, x, y, color in buttons:
        button = plt.Rectangle((x - 1, y - 0.2), 2, 0.4, fill=True, color=color, alpha=1,
                             edgecolor='#dddddd', linewidth=1, zorder=3)
        ax.add_patch(button)
        ax.text(x, y, text, color='white', fontsize=8, ha='center', va='center', fontweight='bold')

    # Add ethical transparency panel at the bottom
    ethical_panel = plt.Rectangle((0.5, 0.5), 9, 1, fill=True, color='#f0f5ff', alpha=1,
                                edgecolor='#bbccdd', linewidth=1, zorder=3)
    ax.add_patch(ethical_panel)
    ax.text(5, 1.25, "ETHICAL AI DISCLOSURE", color='#2c3e50', fontsize=10, ha='center', va='center', fontweight='bold')

    ethical_text = "This response uses trustworthiness filtering (60/100) and consistency verification (65/100). 3 out of 5 sources agree on these treatments. Some experimental treatments showed conflicting evidence."

    ax.text(5, 0.85, ethical_text, color='#2c3e50', fontsize=8, ha='center', va='center', linespacing=1.5, wrap=True)

    plt.tight_layout()
    plt.savefig('ethical_ui_mockup.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("Ethical UI Mockup visualization saved as 'ethical_ui_mockup.png'")

# Create all visualizations with sample data
if __name__ == "__main__":
    # Sample domain results for testing
    sample_domain_results = {
        'diabetes': {
            'standard_rag': {
                'avg_retrieval_time': 0.25,
                'avg_retrieved_docs': 5.2,
                'avg_trustworthiness_avg': 0.65
            },
            'ethical_rag': {
                'avg_retrieval_time': 0.45,
                'avg_retrieved_docs': 4.8,
                'avg_trustworthiness_avg': 0.82,
                'avg_consistency': 0.72,
                'warning_rate': 0.2,
                'hallucination_risks': {'Low': 8, 'Medium': 2, 'Medium-High': 1, 'High': 0},
                'consistency_scores': [0.68, 0.72, 0.75, 0.65, 0.78]
            }
        },
        'cardiovascular': {
            'standard_rag': {
                'avg_retrieval_time': 0.22,
                'avg_retrieved_docs': 5.5,
                'avg_trustworthiness_avg': 0.62
            },
            'ethical_rag': {
                'avg_retrieval_time': 0.42,
                'avg_retrieved_docs': 5.0,
                'avg_trustworthiness_avg': 0.80,
                'avg_consistency': 0.68,
                'warning_rate': 0.3,
                'hallucination_risks': {'Low': 7, 'Medium': 2, 'Medium-High': 2, 'High': 1},
                'consistency_scores': [0.65, 0.68, 0.71, 0.62, 0.74]
            }
        },
        'neurology': {
            'standard_rag': {
                'avg_retrieval_time': 0.28,
                'avg_retrieved_docs': 4.8,
                'avg_trustworthiness_avg': 0.58
            },
            'ethical_rag': {
                'avg_retrieval_time': 0.48,
                'avg_retrieved_docs': 4.2,
                'avg_trustworthiness_avg': 0.78,
                'avg_consistency': 0.62,
                'warning_rate': 0.4,
                'hallucination_risks': {'Low': 6, 'Medium': 3, 'Medium-High': 2, 'High': 1},
                'consistency_scores': [0.58, 0.62, 0.65, 0.60, 0.70]
            }
        },
        'general_medicine': {
            'standard_rag': {
                'avg_retrieval_time': 0.21,
                'avg_retrieved_docs': 6.0,
                'avg_trustworthiness_avg': 0.70
            },
            'ethical_rag': {
                'avg_retrieval_time': 0.41,
                'avg_retrieved_docs': 5.5,
                'avg_trustworthiness_avg': 0.85,
                'avg_consistency': 0.75,
                'warning_rate': 0.15,
                'hallucination_risks': {'Low': 9, 'Medium': 1, 'Medium-High': 0, 'High': 0},
                'consistency_scores': [0.72, 0.75, 0.78, 0.73, 0.80]
            }
        }
    }

    # Set up colors
    viz_colors = {
        'standard_rag': '#1E65B6',  # Blue
        'ethical_rag': '#56A55C',   # Green
        'warning': '#E49B0F',       # Orange
        'risk': '#D7373B',          # Red
        'low_risk': '#56A55C',      # Green
        'medium_risk': '#E4B80F',   # Yellow
        'medium_high_risk': '#E49B0F', # Orange
        'high_risk': '#D7373B'      # Red
    }

    # Create all visualizations
    create_visualizations(sample_domain_results, None, None)

Ethical RAG Dashboard visualization saved as 'ethical_rag_dashboard.png'
Trustworthiness Analysis visualization saved as 'trustworthiness_analysis.png'
Hallucination Prevention visualization saved as 'hallucination_prevention.png'
System Architecture visualization saved as 'ethical_rag_architecture.png'
Ethical UI Mockup visualization saved as 'ethical_ui_mockup.png'


<Figure size 1200x1000 with 0 Axes>